# ML Training with MLflow
Train sklearn models, log experiments to MLflow, artifacts in MinIO.

In [ ]:
import os
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import mlflow
import mlflow.sklearn

tracking_uri = os.getenv('MLFLOW_TRACKING_URI', 'http://localhost:5000')
mlflow.set_tracking_uri(tracking_uri)

s3_endpoint = os.getenv('MLFLOW_S3_ENDPOINT_URL', os.getenv('MINIO_ENDPOINT', 'http://localhost:9000'))
os.environ['MLFLOW_S3_ENDPOINT_URL'] = s3_endpoint
os.environ['AWS_ACCESS_KEY_ID'] = os.getenv('MINIO_ACCESS_KEY', 'minioadmin')
os.environ['AWS_SECRET_ACCESS_KEY'] = os.getenv('MINIO_SECRET_KEY', 'minioadmin')

print(f'MLflow tracking URI: {tracking_uri}')
print(f'Artifact S3 endpoint: {s3_endpoint}')

In [ ]:
import boto3
from botocore.client import Config
import io

s3 = boto3.client(
    's3',
    endpoint_url=os.getenv('MINIO_ENDPOINT', 'http://localhost:9000'),
    aws_access_key_id=os.getenv('MINIO_ACCESS_KEY', 'minioadmin'),
    aws_secret_access_key=os.getenv('MINIO_SECRET_KEY', 'minioadmin'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

bucket = 'training-data'
existing = [b['Name'] for b in s3.list_buckets()['Buckets']]

if bucket in existing:
    objs = s3.list_objects_v2(Bucket=bucket).get('Contents', [])
    parquet_keys = [o['Key'] for o in objs if o['Key'].endswith('.parquet')]
    if parquet_keys:
        obj = s3.get_object(Bucket=bucket, Key=parquet_keys[0])
        df = pd.read_parquet(io.BytesIO(obj['Body'].read()))
        print(f'Loaded {len(df)} rows from s3://{bucket}/{parquet_keys[0]}')
    else:
        df = None
else:
    df = None

if df is None:
    print('No training data found — using synthetic ecommerce orders dataset')
    np.random.seed(42)
    n = 5000
    df = pd.DataFrame({
        'quantity': np.random.randint(1, 20, n),
        'unit_price': np.random.uniform(5, 500, n),
        'discount_pct': np.random.uniform(0, 0.4, n),
        'customer_age': np.random.randint(18, 70, n),
        'days_to_ship': np.random.randint(1, 14, n),
        'order_value': None
    })
    df['order_value'] = (
        df['quantity'] * df['unit_price'] * (1 - df['discount_pct'])
        + np.random.normal(0, 10, n)
    )
    print(df.head())

In [ ]:
feature_cols = [c for c in df.columns if c != 'order_value']
X = df[feature_cols].select_dtypes(include='number')
y = df['order_value']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train size: {len(X_train)}, Test size: {len(X_test)}')
print(f'Features: {X.columns.tolist()}')

In [ ]:
mlflow.set_experiment('ecommerce-order-prediction')

with mlflow.start_run(run_name='random-forest') as run:
    params = {'n_estimators': 100, 'max_depth': 6, 'random_state': 42}
    mlflow.log_params(params)

    rf = RandomForestRegressor(**params)
    rf.fit(X_train, y_train)
    preds = rf.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('r2', r2)

    mlflow.sklearn.log_model(rf, 'model')
    rf_run_id = run.info.run_id

print(f'RandomForest — MAE: {mae:.2f}, R2: {r2:.4f}')
print(f'Run ID: {rf_run_id}')

In [ ]:
with mlflow.start_run(run_name='gradient-boosting') as run:
    params = {'n_estimators': 200, 'learning_rate': 0.05, 'max_depth': 4, 'random_state': 42}
    mlflow.log_params(params)

    gb = GradientBoostingRegressor(**params)
    gb.fit(X_train, y_train)
    preds = gb.predict(X_test)

    mae = mean_absolute_error(y_test, preds)
    r2 = r2_score(y_test, preds)
    mlflow.log_metric('mae', mae)
    mlflow.log_metric('r2', r2)

    mlflow.sklearn.log_model(gb, 'model')
    gb_run_id = run.info.run_id

print(f'GradientBoosting — MAE: {mae:.2f}, R2: {r2:.4f}')
print(f'Run ID: {gb_run_id}')

In [ ]:
runs = mlflow.search_runs(
    experiment_names=['ecommerce-order-prediction'],
    order_by=['metrics.r2 DESC']
)
display(runs[['run_id', 'tags.mlflow.runName', 'metrics.mae', 'metrics.r2', 'status']])

Check MLflow UI to see your experiments and model artifacts stored in MinIO!